# Data Consumption — KPI queries (exploitation Delta)

Define cymatics / audio-metadata KPIs and run them against the **exploitation-zone** Delta table
(`s3a://<EXPLOITATION_ZONE_BUCKET>/metadata/observations_delta/`).

Prerequisites: MinIO up, trusted + exploitation processing completed (Parquet + Delta sync).

See also: [`delta_lake_discovery.ipynb`](delta_lake_discovery.ipynb) for table health checks.

## Paths and environment

In [4]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
_root = next(
    (p for p in [_here, *_here.parents] if (p / "docker-compose.yml").is_file() and (p / "orchestrate.py").is_file()),
    None,
)
if _root is None:
    raise RuntimeError("Repo root not found — open the notebook from the BDM-Cymatics tree")

PROJECT_ROOT = str(_root)
DATA_CONSUMPTION_DIR = _root / "data_consumption"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
if str(DATA_CONSUMPTION_DIR) not in sys.path:
    sys.path.insert(1, str(DATA_CONSUMPTION_DIR))

from dotenv import load_dotenv
load_dotenv(_root / ".env", override=False)

EXPLOITATION_BUCKET = os.environ.get("EXPLOITATION_ZONE_BUCKET", "exploitation-zone")
print("PROJECT_ROOT =", PROJECT_ROOT)
print("EXPLOITATION_BUCKET =", EXPLOITATION_BUCKET)

PROJECT_ROOT = /Users/arman/Desktop/2026-sprinrg/BDM/Cymatics/BDM-Cymatics
EXPLOITATION_BUCKET = exploitation-zone


## Load exploitation-zone Delta table

In [12]:
from deltalake import DeltaTable
from shared.sync_delta import OBSERVATIONS_DELTA_PATH, s3_storage_options

STORAGE_OPTIONS = s3_storage_options()
EXPLOITATION_DELTA_URI = f"s3a://{EXPLOITATION_BUCKET}/{OBSERVATIONS_DELTA_PATH}"

dt = DeltaTable(EXPLOITATION_DELTA_URI, storage_options=STORAGE_OPTIONS)
obs = dt.to_pandas()

print(f"Loaded {len(obs)} row(s) from {EXPLOITATION_DELTA_URI}")
print(f"Delta version: {dt.version()}")

Loaded 22 row(s) from s3a://exploitation-zone/metadata/observations_delta
Delta version: 0


## KPI registry

Add new KPIs to `KPI_REGISTRY` (id → title + description + runner). Each runner returns a DataFrame.

In [13]:
import pandas as pd
import numpy as np
from itertools import combinations


def _clean_category(series: pd.Series) -> pd.Series:
    s = series.fillna("").astype(str).str.strip()
    return s.mask(s.isin(["", "-", "nan", "None"]), np.nan)


def _clean_source(series: pd.Series) -> pd.Series:
    s = series.fillna("").astype(str).str.strip()
    return s.mask(s.isin(["", "-", "nan", "None"]), np.nan)


def _clean_peak_frequency_hz(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def _frequency_band(hz: float) -> str:
    if hz < 200:
        return "Low (0-200 Hz)"
    if hz < 1000:
        return "Mid (200-1000 Hz)"
    return "High (1000+ Hz)"


def kpi_most_repeated_frequency_per_category(df: pd.DataFrame) -> pd.DataFrame:
    """KPI 1 — Most repeated peak frequency (Hz) within each category."""
    work = df.copy()
    work["category"] = _clean_category(work["category"])
    work["peak_frequency_hz"] = _clean_peak_frequency_hz(work["peak_frequency_hz"])
    work = work.dropna(subset=["category", "peak_frequency_hz"])
    if work.empty:
        return pd.DataFrame(
            columns=[
                "category",
                "most_repeated_frequency_hz",
                "occurrence_count",
                "total_observations_in_category",
                "share_within_category",
            ]
        )

    work["frequency_hz_rounded"] = work["peak_frequency_hz"].round().astype(int)
    freq_counts = (
        work.groupby(["category", "frequency_hz_rounded"], as_index=False)
        .size()
        .rename(columns={"size": "occurrence_count"})
    )
    totals = work.groupby("category").size().rename("total_observations_in_category")
    idx = freq_counts.groupby("category")["occurrence_count"].idxmax()
    top = freq_counts.loc[idx].rename(
        columns={"frequency_hz_rounded": "most_repeated_frequency_hz"}
    )
    top = top.merge(totals, on="category", how="left")
    top["share_within_category"] = (
        top["occurrence_count"] / top["total_observations_in_category"]
    ).round(4)
    return top.sort_values(["category", "most_repeated_frequency_hz"]).reset_index(drop=True)[
        [
            "category",
            "most_repeated_frequency_hz",
            "occurrence_count",
            "total_observations_in_category",
            "share_within_category",
        ]
    ]


def kpi_best_cymatics_candidates(df: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    """KPI 2 — Top recordings by combined symmetry + pattern stability (dataset-wide)."""
    work = df.copy()
    for col in ("symmetry_score", "pattern_stability_score"):
        work[col] = pd.to_numeric(work[col], errors="coerce")
    work = work.dropna(subset=["symmetry_score", "pattern_stability_score"])
    if work.empty:
        return pd.DataFrame()

    work["combined_cymatics_score"] = (
        work["symmetry_score"] + work["pattern_stability_score"]
    ) / 2.0
    work["rank"] = work["combined_cymatics_score"].rank(
        method="dense", ascending=False
    ).astype(int)

    cols = [
        "rank",
        "uuid",
        "source_id",
        "category",
        "source",
        "symmetry_score",
        "pattern_stability_score",
        "combined_cymatics_score",
        "peak_frequency_hz",
        "audio_path",
    ]
    cols = [c for c in cols if c in work.columns]
    return (
        work.nlargest(top_n, "combined_cymatics_score")
        .loc[:, cols]
        .reset_index(drop=True)
    )


def kpi_frequency_clusters(df: pd.DataFrame) -> pd.DataFrame:
    """KPI 3 — Recording and category counts per Low / Mid / High frequency band."""
    work = df.copy()
    work["peak_frequency_hz"] = _clean_peak_frequency_hz(work["peak_frequency_hz"])
    work["category"] = _clean_category(work.get("category", pd.Series(dtype=object)))
    work = work.dropna(subset=["peak_frequency_hz"])
    if work.empty:
        return pd.DataFrame(
            columns=[
                "frequency_band",
                "recording_count",
                "distinct_category_count",
                "share_of_recordings",
            ]
        )

    work["frequency_band"] = work["peak_frequency_hz"].map(_frequency_band)
    total = len(work)
    out = (
        work.groupby("frequency_band", as_index=False)
        .agg(
            recording_count=("uuid", "count"),
            distinct_category_count=("category", lambda s: s.dropna().nunique()),
        )
    )
    out["share_of_recordings"] = (out["recording_count"] / total).round(4)
    band_order = ["Low (0-200 Hz)", "Mid (200-1000 Hz)", "High (1000+ Hz)"]
    out["frequency_band"] = pd.Categorical(out["frequency_band"], categories=band_order, ordered=True)
    return out.sort_values("frequency_band").reset_index(drop=True)


def kpi_most_similar_frequency_categories(
    df: pd.DataFrame, top_pairs: int = 10
) -> pd.DataFrame:
    """KPI 4 — Category pairs with closest average peak frequency."""
    work = df.copy()
    work["category"] = _clean_category(work["category"])
    work["peak_frequency_hz"] = _clean_peak_frequency_hz(work["peak_frequency_hz"])
    work = work.dropna(subset=["category", "peak_frequency_hz"])
    if work.empty:
        return pd.DataFrame()

    cat_mean = (
        work.groupby("category", as_index=False)["peak_frequency_hz"]
        .mean()
        .rename(columns={"peak_frequency_hz": "avg_peak_frequency_hz"})
    )
    if len(cat_mean) < 2:
        return pd.DataFrame()

    means = cat_mean.set_index("category")["avg_peak_frequency_hz"]
    pairs = []
    for c1, c2 in combinations(cat_mean["category"].tolist(), 2):
        pairs.append(
            {
                "category_a": c1,
                "category_b": c2,
                "avg_peak_frequency_hz_a": round(means[c1], 2),
                "avg_peak_frequency_hz_b": round(means[c2], 2),
                "frequency_difference_hz": round(abs(means[c1] - means[c2]), 2),
            }
        )
    result = pd.DataFrame(pairs).sort_values(
        "frequency_difference_hz", ascending=True
    ).reset_index(drop=True)
    result.insert(0, "rank", range(1, len(result) + 1))
    return result.head(top_pairs)


def kpi_avg_processing_time_per_source(df: pd.DataFrame) -> pd.DataFrame:
    """KPI 5 — Mean trusted-zone processing time (seconds) by landing source path."""
    work = df.copy()
    work["source"] = _clean_source(work["source"])
    work["processing_time(seconds)"] = pd.to_numeric(
        work["processing_time(seconds)"], errors="coerce"
    )
    work = work.dropna(subset=["source", "processing_time(seconds)"])
    if work.empty:
        return pd.DataFrame()

    out = (
        work.groupby("source", as_index=False)
        .agg(
            avg_processing_time_seconds=("processing_time(seconds)", "mean"),
            std_processing_time_seconds=("processing_time(seconds)", "std"),
            min_processing_time_seconds=("processing_time(seconds)", "min"),
            max_processing_time_seconds=("processing_time(seconds)", "max"),
            recording_count=("uuid", "count"),
        )
    )
    for col in out.columns:
        if col.endswith("_seconds"):
            out[col] = out[col].round(3)
    return out.sort_values("avg_processing_time_seconds", ascending=False).reset_index(drop=True)


def kpi_most_complex_sounds(df: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    """KPI 6 — Highest spectral_entropy recordings (noise-like / rich harmonics)."""
    work = df.copy()
    work["spectral_entropy"] = pd.to_numeric(work["spectral_entropy"], errors="coerce")
    work = work.dropna(subset=["spectral_entropy"])
    if work.empty:
        return pd.DataFrame()

    cols = [
        "uuid",
        "category",
        "source",
        "spectral_entropy",
        "peak_frequency_hz",
        "spectral_flatness",
        "audio_path",
    ]
    cols = [c for c in cols if c in work.columns]
    top = work.nlargest(top_n, "spectral_entropy").loc[:, cols].reset_index(drop=True)
    top.insert(0, "rank", range(1, len(top) + 1))
    return top


def kpi_brightest_darkest_sounds(
    df: pd.DataFrame, top_n: int = 10
) -> pd.DataFrame:
    """KPI 7 — Brightest / darkest recordings and category averages by spectral_centroid_hz."""
    work = df.copy()
    work["spectral_centroid_hz"] = pd.to_numeric(work["spectral_centroid_hz"], errors="coerce")
    work = work.dropna(subset=["spectral_centroid_hz"])
    if work.empty:
        return pd.DataFrame()

    parts = []
    rec_cols = ["uuid", "category", "source", "spectral_centroid_hz", "peak_frequency_hz", "audio_path"]
    rec_cols = [c for c in rec_cols if c in work.columns]

    for rank_group, take_largest in (
        ("brightest_recording", True),
        ("darkest_recording", False),
    ):
        sub = (
            work.nlargest(top_n, "spectral_centroid_hz")
            if take_largest
            else work.nsmallest(top_n, "spectral_centroid_hz")
        )
        sub = sub.loc[:, rec_cols].copy()
        sub["aggregation_level"] = "recording"
        sub["rank_group"] = rank_group
        parts.append(sub)

    cat_work = work.copy()
    cat_work["category"] = _clean_category(cat_work["category"])
    cat_work = cat_work.dropna(subset=["category"])
    if not cat_work.empty:
        cat_avg = (
            cat_work.groupby("category", as_index=False)["spectral_centroid_hz"]
            .mean()
            .rename(columns={"spectral_centroid_hz": "avg_spectral_centroid_hz"})
        )
        for rank_group, take_largest in (
            ("brightest_category_avg", True),
            ("darkest_category_avg", False),
        ):
            sub = (
                cat_avg.nlargest(top_n, "avg_spectral_centroid_hz")
                if take_largest
                else cat_avg.nsmallest(top_n, "avg_spectral_centroid_hz")
            ).copy()
            sub["aggregation_level"] = "category"
            sub["rank_group"] = rank_group
            sub["uuid"] = np.nan
            sub["source"] = np.nan
            sub["audio_path"] = np.nan
            sub["peak_frequency_hz"] = np.nan
            sub["spectral_centroid_hz"] = sub["avg_spectral_centroid_hz"]
            sub = sub.drop(columns=["avg_spectral_centroid_hz"])
            parts.append(sub)

    out = pd.concat(parts, ignore_index=True, sort=False)
    out.insert(0, "rank", range(1, len(out) + 1))
    return out


KPI_REGISTRY = {
    "kpi_01_most_repeated_frequency_per_category": {
        "title": "Most repeated peak frequency per category",
        "description": (
            "For each non-empty category, the peak frequency (Hz, rounded to integer) "
            "that appears on the most observations."
        ),
        "runner": kpi_most_repeated_frequency_per_category,
    },
    "kpi_02_best_cymatics_candidates": {
        "title": "Best cymatics candidates (top 10 symmetry + stability)",
        "description": (
            "Top 10 individual recordings with the highest combined cymatics score "
            "(average of symmetry_score and pattern_stability_score) across the full dataset."
        ),
        "runner": kpi_best_cymatics_candidates,
    },
    "kpi_03_frequency_clusters": {
        "title": "Frequency clusters (low / mid / high distribution)",
        "description": (
            "Groups recordings into Low (0–200 Hz), Mid (200–1000 Hz), and High (1000+ Hz) "
            "and counts recordings plus distinct categories per band."
        ),
        "runner": kpi_frequency_clusters,
    },
    "kpi_04_most_similar_frequency_categories": {
        "title": "Most similar frequency categories",
        "description": (
            "Category pairs with the smallest difference in average peak_frequency_hz — "
            "sounds most likely to be confused in frequency space."
        ),
        "runner": kpi_most_similar_frequency_categories,
    },
    "kpi_05_avg_processing_time_per_source": {
        "title": "Average processing time per source",
        "description": (
            "Mean trusted-zone processing_time(seconds) per landing source "
            "(warm-path, hot-path, cold-path) — cymatics image + video generation."
        ),
        "runner": kpi_avg_processing_time_per_source,
    },
    "kpi_06_most_complex_sounds": {
        "title": "Most complex sounds (spectral entropy)",
        "description": (
            "Top 10 recordings ranked by spectral_entropy (high = complex / noise-like; "
            "low = purer tones)."
        ),
        "runner": kpi_most_complex_sounds,
    },
    "kpi_07_brightest_darkest_sounds": {
        "title": "Brightest vs darkest sounds (spectral centroid)",
        "description": (
            "Top 10 brightest and darkest recordings by spectral_centroid_hz, plus "
            "category-level averages for the same metric."
        ),
        "runner": kpi_brightest_darkest_sounds,
    },
}

for kpi_id, meta in KPI_REGISTRY.items():
    print(f"  {kpi_id}: {meta['title']}")

  kpi_01_most_repeated_frequency_per_category: Most repeated peak frequency per category
  kpi_02_best_cymatics_candidates: Best cymatics candidates (top 10 symmetry + stability)
  kpi_03_frequency_clusters: Frequency clusters (low / mid / high distribution)
  kpi_04_most_similar_frequency_categories: Most similar frequency categories
  kpi_05_avg_processing_time_per_source: Average processing time per source
  kpi_06_most_complex_sounds: Most complex sounds (spectral entropy)
  kpi_07_brightest_darkest_sounds: Brightest vs darkest sounds (spectral centroid)


---

## KPI 1 — Most repeated peak frequency per category

**Question:** Within each sound category, which dominant peak frequency (Hz) occurs most often?

**Source columns:** `category`, `peak_frequency_hz`

In [14]:
KPI_ID = "kpi_01_most_repeated_frequency_per_category"
meta = KPI_REGISTRY[KPI_ID]
print(meta["title"])
print(meta["description"])
print()
kpi_01_result = meta["runner"](obs)
if kpi_01_result.empty:
    print("No rows with both category and peak_frequency_hz.")
else:
    display(kpi_01_result)
    print(f"\nCategories covered: {kpi_01_result['category'].nunique()}")

Most repeated peak frequency per category
For each non-empty category, the peak frequency (Hz, rounded to integer) that appears on the most observations.



,category,most_repeated_frequency_hz,occurrence_count,total_observations_in_category,share_within_category
0,bird song,3572,1,2,0.50
1,cat purring,56,1,2,0.50
2,church bells,540,1,2,0.50
3,crickets,116,1,1,1.00
4,forest,32,1,1,1.00
5,ocean,4,1,1,1.00
6,rain,516,1,1,1.00
7,sea waves,184,1,1,1.00
8,singing bowl,540,1,2,0.50
9,tuning fork,92,1,4,0.25



Categories covered: 12


### KPI 1 — frequency distribution per category (detail)

Top frequencies per category (all bins, not only the winner).

In [15]:
DETAIL_CATEGORY = ""  # e.g. "rain" or "dog"; leave empty for all
TOP_N = 5

work = obs.copy()
work["category"] = _clean_category(work["category"])
work["peak_frequency_hz"] = _clean_peak_frequency_hz(work["peak_frequency_hz"])
work = work.dropna(subset=["category", "peak_frequency_hz"])
work["frequency_hz_rounded"] = work["peak_frequency_hz"].round().astype(int)

if DETAIL_CATEGORY.strip():
    work = work[work["category"].str.contains(DETAIL_CATEGORY.strip(), case=False, na=False)]

if work.empty:
    print("No matching rows for detail view.")
else:
    detail = (
        work.groupby(["category", "frequency_hz_rounded"], as_index=False)
        .size()
        .rename(columns={"size": "occurrence_count"})
    )
    detail["rank_in_category"] = detail.groupby("category")["occurrence_count"].rank(
        method="dense", ascending=False
    )
    detail = detail[detail["rank_in_category"] <= TOP_N].sort_values(
        ["category", "rank_in_category", "frequency_hz_rounded"]
    )
    display(detail)

,category,frequency_hz_rounded,occurrence_count,rank_in_category
0,bird song,3572,1,1.0
1,bird song,3732,1,1.0
2,cat purring,56,1,1.0
3,cat purring,116,1,1.0
4,church bells,540,1,1.0
5,church bells,1480,1,1.0
6,crickets,116,1,1.0
7,forest,32,1,1.0
8,ocean,4,1,1.0
9,rain,516,1,1.0


---

## KPI 2 — Best cymatics candidates (top 10)

**Question:** Which individual recordings produce the most visually striking cymatics patterns?

**Score:** `(symmetry_score + pattern_stability_score) / 2` — ranked dataset-wide (not per category).

In [16]:
KPI_ID = "kpi_02_best_cymatics_candidates"
meta = KPI_REGISTRY[KPI_ID]
print(meta["title"])
print(meta["description"])
print()
kpi_02_result = meta["runner"](obs)
if kpi_02_result.empty:
    print("(no data)")
else:
    display(kpi_02_result)

Best cymatics candidates (top 10 symmetry + stability)
Top 10 individual recordings with the highest combined cymatics score (average of symmetry_score and pattern_stability_score) across the full dataset.



,rank,uuid,source_id,category,source,symmetry_score,pattern_stability_score,combined_cymatics_score,peak_frequency_hz,audio_path
0,1,fe8e60d9-14b4-484a-a7f6-4e9d5d6a9dc2,361922,tuning fork,Freesound,0.965683,0.988388,0.977035,440.0,audio/440/fe8e60d9-14b4-484a-a7f6-4e9d5d6a9dc2...
1,2,6eda6370-90ad-4b6e-8aa0-4971155d9975,740418,tuning fork,Freesound,0.967929,0.983545,0.975737,96.0,audio/96/6eda6370-90ad-4b6e-8aa0-4971155d9975-...
2,3,f736c6c1-4601-4db6-97e8-a01eebccb2a5,420308,wind,Freesound,0.960284,0.985796,0.973040,20.0,audio/20/f736c6c1-4601-4db6-97e8-a01eebccb2a5-...
3,4,00a11728-db7d-44b1-912f-1d2e4b0899c8,485373,cat purring,Freesound,0.959478,0.982892,0.971185,56.0,audio/56/00a11728-db7d-44b1-912f-1d2e4b0899c8-...
4,5,60d4baa2-d190-4196-afdd-51b160178e62,96146,church bells,Freesound,0.964095,0.973764,0.968929,540.0,audio/540/60d4baa2-d190-4196-afdd-51b160178e62...
5,6,5706b0f9-a9d9-4cd3-bbe2-668eec60d5c1,507265,bird song,Freesound,0.963236,0.971764,0.967500,3732.0,audio/3732/5706b0f9-a9d9-4cd3-bbe2-668eec60d5c...
6,7,75600263-c041-4b98-bba7-21af543be528,740185,tuning fork,Freesound,0.954406,0.974842,0.964624,92.0,audio/92/75600263-c041-4b98-bba7-21af543be528-...
7,8,3d6ab8ab-b4a3-4466-bbf4-c34380266bc9,810426,singing bowl,Freesound,0.960470,0.961909,0.961190,540.0,audio/540/3d6ab8ab-b4a3-4466-bbf4-c34380266bc9...
8,9,b637aadd-515b-4fc6-ab72-4cd5f9508180,507256,bird song,Freesound,0.959306,0.960873,0.960090,3572.0,audio/3572/b637aadd-515b-4fc6-ab72-4cd5f950818...
9,10,13159abb-f78e-48a8-93fb-9157e02475d7,809062,tuning fork,Freesound,0.956626,0.961403,0.959015,6272.0,audio/6272/13159abb-f78e-48a8-93fb-9157e02475d...


---

## KPI 3 — Frequency clusters (low / mid / high)

**Bands:** Low `0–200 Hz`, Mid `200–1000 Hz`, High `≥1000 Hz`.

Reports recording counts, distinct categories per band, and share of corpus.

In [17]:
KPI_ID = "kpi_03_frequency_clusters"
meta = KPI_REGISTRY[KPI_ID]
print(meta["title"])
print(meta["description"])
print()
kpi_03_result = meta["runner"](obs)
if kpi_03_result.empty:
    print("(no data)")
else:
    display(kpi_03_result)

Frequency clusters (low / mid / high distribution)
Groups recordings into Low (0–200 Hz), Mid (200–1000 Hz), and High (1000+ Hz) and counts recordings plus distinct categories per band.



,frequency_band,recording_count,distinct_category_count,share_of_recordings
0,Low (0-200 Hz),13,8,0.5909
1,Mid (200-1000 Hz),4,4,0.1818
2,High (1000+ Hz),5,4,0.2273


---

## KPI 4 — Most similar frequency categories

**Question:** Which category pairs have the closest average `peak_frequency_hz`?

Useful for spotting categories that may be confused by listeners or ML models.

In [18]:
KPI_ID = "kpi_04_most_similar_frequency_categories"
meta = KPI_REGISTRY[KPI_ID]
print(meta["title"])
print(meta["description"])
print()
kpi_04_result = meta["runner"](obs)
if kpi_04_result.empty:
    print("Need at least two categories with peak_frequency_hz.")
else:
    display(kpi_04_result)

Most similar frequency categories
Category pairs with the smallest difference in average peak_frequency_hz — sounds most likely to be confused in frequency space.



,rank,category_a,category_b,avg_peak_frequency_hz_a,avg_peak_frequency_hz_b,frequency_difference_hz
0,1,forest,wind,32.0,20.0,12.0
1,2,ocean,wind,4.0,20.0,16.0
2,3,forest,waterfall,32.0,56.0,24.0
3,4,forest,ocean,32.0,4.0,28.0
4,5,cat purring,waterfall,86.0,56.0,30.0
5,6,cat purring,crickets,86.0,116.0,30.0
6,7,waterfall,wind,56.0,20.0,36.0
7,8,ocean,waterfall,4.0,56.0,52.0
8,9,cat purring,forest,86.0,32.0,54.0
9,10,crickets,waterfall,116.0,56.0,60.0


---

## KPI 5 — Average processing time per source

**Question:** How long does trusted-zone cymatics processing take per landing source?

**Source column:** `processing_time(seconds)` (trusted pipeline) grouped by `source` (`warm-path`, `hot-path`, `cold-path`, …).

In [19]:
KPI_ID = "kpi_05_avg_processing_time_per_source"
meta = KPI_REGISTRY[KPI_ID]
print(meta["title"])
print(meta["description"])
print()
kpi_05_result = meta["runner"](obs)
if kpi_05_result.empty:
    print("(no data)")
else:
    display(kpi_05_result)

Average processing time per source
Mean trusted-zone processing_time(seconds) per landing source (warm-path, hot-path, cold-path) — cymatics image + video generation.



,source,avg_processing_time_seconds,std_processing_time_seconds,min_processing_time_seconds,max_processing_time_seconds,recording_count
0,hot-path,12.459,0.903,11.821,13.098,2
1,Freesound,10.853,3.125,6.491,14.729,20


---

## KPI 6 — Most complex sounds (spectral entropy)

**Interpretation:** High `spectral_entropy` → complex, noise-like, rich harmonics. Low → purer tones.

In [20]:
KPI_ID = "kpi_06_most_complex_sounds"
meta = KPI_REGISTRY[KPI_ID]
print(meta["title"])
print(meta["description"])
print()
kpi_06_result = meta["runner"](obs)
if kpi_06_result.empty:
    print("(no data — run exploitation Spark batch first)")
else:
    display(kpi_06_result)

Most complex sounds (spectral entropy)
Top 10 recordings ranked by spectral_entropy (high = complex / noise-like; low = purer tones).



,rank,uuid,category,source,spectral_entropy,peak_frequency_hz,spectral_flatness,audio_path
0,1,c7f54155-12af-424f-87e0-37990f7a4ffc,forest,Freesound,10.757884,32.0,0.3130275763765992,audio/32/c7f54155-12af-424f-87e0-37990f7a4ffc-...
1,2,09bfde40-7dfb-46bd-a2bf-df18007ec9ed,sea waves,Freesound,9.799672,184.0,0.18480683257430908,audio/184/09bfde40-7dfb-46bd-a2bf-df18007ec9ed...
2,3,c45ba243-340b-4412-a31c-5686c68c769b,crickets,Freesound,9.695339,116.0,0.41521724581813557,audio/116/c45ba243-340b-4412-a31c-5686c68c769b...
3,4,3bbf9f45-8666-4d97-981e-3b0cf835f5fa,singing bowl,Freesound,9.464582,1308.0,0.2442409093755489,audio/1308/3bbf9f45-8666-4d97-981e-3b0cf835f5f...
4,5,31dac2a1-65a2-439d-a958-3574fef224d0,rain,Freesound,9.389345,516.0,0.062251579491353445,audio/516/31dac2a1-65a2-439d-a958-3574fef224d0...
5,6,a6899d2e-efd5-4007-b7e6-84ee3576c530,waterfall,Freesound,8.999524,44.0,0.04667301330665712,audio/44/a6899d2e-efd5-4007-b7e6-84ee3576c530-...
6,7,a2f5f95b-8804-43c0-85fb-af6834d2f91f,ocean,Freesound,8.659733,4.0,0.0857102187216942,audio/4/a2f5f95b-8804-43c0-85fb-af6834d2f91f-4...
7,8,e1ffd9d3-5cd0-4857-bfa5-49df4c0f55aa,waterfall,Freesound,8.613555,68.0,0.19451831127968444,audio/68/e1ffd9d3-5cd0-4857-bfa5-49df4c0f55aa-...
8,9,13159abb-f78e-48a8-93fb-9157e02475d7,tuning fork,Freesound,8.566127,6272.0,0.4284869609440066,audio/6272/13159abb-f78e-48a8-93fb-9157e02475d...
9,10,b637aadd-515b-4fc6-ab72-4cd5f9508180,bird song,Freesound,7.863215,3572.0,0.009408794840358321,audio/3572/b637aadd-515b-4fc6-ab72-4cd5f950818...


---

## KPI 7 — Brightest vs darkest sounds (spectral centroid)

**Interpretation:** Higher `spectral_centroid_hz` → brighter timbre. Includes top/bottom recordings and category averages.

In [21]:
KPI_ID = "kpi_07_brightest_darkest_sounds"
meta = KPI_REGISTRY[KPI_ID]
print(meta["title"])
print(meta["description"])
print()
kpi_07_result = meta["runner"](obs)
if kpi_07_result.empty:
    print("(no data — run exploitation Spark batch first)")
else:
    display(kpi_07_result)

Brightest vs darkest sounds (spectral centroid)
Top 10 brightest and darkest recordings by spectral_centroid_hz, plus category-level averages for the same metric.



,rank,uuid,category,source,spectral_centroid_hz,peak_frequency_hz,audio_path,aggregation_level,rank_group
0,1,13159abb-f78e-48a8-93fb-9157e02475d7,tuning fork,Freesound,6523.039636,6272.0,audio/6272/13159abb-f78e-48a8-93fb-9157e02475d...,recording,brightest_recording
1,2,c45ba243-340b-4412-a31c-5686c68c769b,crickets,Freesound,5437.995640,116.0,audio/116/c45ba243-340b-4412-a31c-5686c68c769b...,recording,brightest_recording
2,3,5706b0f9-a9d9-4cd3-bbe2-668eec60d5c1,bird song,Freesound,4856.167221,3732.0,audio/3732/5706b0f9-a9d9-4cd3-bbe2-668eec60d5c...,recording,brightest_recording
3,4,3bbf9f45-8666-4d97-981e-3b0cf835f5fa,singing bowl,Freesound,4740.770832,1308.0,audio/1308/3bbf9f45-8666-4d97-981e-3b0cf835f5f...,recording,brightest_recording
4,5,c7f54155-12af-424f-87e0-37990f7a4ffc,forest,Freesound,4617.679771,32.0,audio/32/c7f54155-12af-424f-87e0-37990f7a4ffc-...,recording,brightest_recording
5,6,b637aadd-515b-4fc6-ab72-4cd5f9508180,bird song,Freesound,3716.525750,3572.0,audio/3572/b637aadd-515b-4fc6-ab72-4cd5f950818...,recording,brightest_recording
6,7,09bfde40-7dfb-46bd-a2bf-df18007ec9ed,sea waves,Freesound,2412.462150,184.0,audio/184/09bfde40-7dfb-46bd-a2bf-df18007ec9ed...,recording,brightest_recording
7,8,75600263-c041-4b98-bba7-21af543be528,tuning fork,Freesound,2259.327747,92.0,audio/92/75600263-c041-4b98-bba7-21af543be528-...,recording,brightest_recording
8,9,6eda6370-90ad-4b6e-8aa0-4971155d9975,tuning fork,Freesound,1833.062173,96.0,audio/96/6eda6370-90ad-4b6e-8aa0-4971155d9975-...,recording,brightest_recording
9,10,60d4baa2-d190-4196-afdd-51b160178e62,church bells,Freesound,1677.917880,540.0,audio/540/60d4baa2-d190-4196-afdd-51b160178e62...,recording,brightest_recording


## Run all registered KPIs

In [ ]:
kpi_outputs = {}

for kpi_id, meta in KPI_REGISTRY.items():
    print("=" * 72)
    print(f"{kpi_id}\n{meta['title']}")
    print(meta["description"])
    result = meta["runner"](obs)
    kpi_outputs[kpi_id] = result
    if result.empty:
        print("(no data)")
    else:
        display(result)
    print()